Real-Time Face Recognition

Built using `face_recognition` (dlib-based) + `OpenCV`, following the standard pipeline:

```
Face Detection → Face Embedding → Compare → Identity
```

> **Note:** Unlike CIFAR-10 image classification (CNN trained on fixed classes), face recognition works by comparing **128-dimensional face embeddings**. No training is required — you just register known faces and the system compares against them.

**Pipeline:**
1. Register known faces from images (one photo per person)
2. Open webcam, detect faces in each frame
3. Generate embeddings and compare against known faces
4. Label and draw boxes around recognized faces in real time

## Step 1: Install Libraries

> **Important:** Run this cell first to install all required dependencies.

In [ ]:
!pip install cmake
!pip install dlib
!pip install face_recognition
!pip install opencv-python
!pip install numpy matplotlib

## Step 2: Import Libraries

In [ ]:
import cv2
import face_recognition
import numpy as np
import os
import matplotlib.pyplot as plt

print('[SUCCESS] Libraries imported successfully!')
print(f'OpenCV version: {cv2.__version__}')
print(f'NumPy version: {np.__version__}')

## Step 3: Set Up Known Faces Folder



In [ ]:
KNOWN_FACES_DIR = './known_faces'

os.makedirs(KNOWN_FACES_DIR, exist_ok=True)

files = [f for f in os.listdir(KNOWN_FACES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f'Found {len(files)} known face image(s):')
for f in files:
    print(f'  - {f}')

if len(files) == 0:
    print('\n[WARNING] Add at least one photo to the known_faces folder before continuing!')

##  Step 4: Generate Face Encodings



In [ ]:
known_encodings = []
known_names = []

for filename in files:
    path = os.path.join(KNOWN_FACES_DIR, filename)
    name = os.path.splitext(filename)[0]

    image = face_recognition.load_image_file(path)
    encodings = face_recognition.face_encodings(image)

    if len(encodings) == 0:
        print(f'[WARNING] No face found in {filename} — skipping.')
        continue

    known_encodings.append(encodings[0])
    known_names.append(name)
    print(f'[SUCCESS] Encoded: {name}')

print(f'\nTotal registered faces: {len(known_names)}')
if known_names:
    print(f'Names: {known_names}')

##  Step 5: Preview Registered Faces

In [ ]:
if len(known_names) > 0:
    fig, axes = plt.subplots(1, len(known_names), figsize=(4 * len(known_names), 4))
    if len(known_names) == 1:
        axes = [axes]

    for ax, filename, name in zip(axes, files, known_names):
        img = face_recognition.load_image_file(os.path.join(KNOWN_FACES_DIR, filename))
        ax.imshow(img)
        ax.set_title(name, fontsize=12, fontweight='bold')
        ax.axis('off')

    plt.suptitle('Registered Known Faces', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No faces to preview yet.')

##  Step 6: Real-Time Face Recognition (Webcam)

> **Important:** This opens a live webcam window. It will **only work when run locally** (VS Code on your machine) — it cannot run in a hosted/cloud notebook.
> 
> - Press **ESC** to quit
> - Press **s** to save a snapshot of the current frame
> 


In [ ]:
def run_realtime_recognition(known_encodings, known_names, threshold=0.6, camera_index=0):
    """Run real-time face recognition using webcam."""

    if len(known_encodings) == 0:
        print('[WARNING] No known faces registered. Add images to known_faces/ first.')
        return

    video = cv2.VideoCapture(camera_index)

    if not video.isOpened():
        print('[ERROR] Could not open webcam. Check camera_index or permissions.')
        return

    print('[INFO] Starting real-time recognition... Press ESC to quit, "s" to save snapshot.')
    snapshot_count = 0

    while True:
        ret, frame = video.read()
        if not ret:
            print('Failed to grab frame.')
            break

        # Resize frame to 1/4 size for faster processing
        small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
        rgb_small_frame = small_frame[:, :, ::-1]   # BGR -> RGB

        # Detect faces and compute encodings
        face_locations = face_recognition.face_locations(rgb_small_frame)
        face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

        for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
            # Scale coordinates back to original frame size
            top, right, bottom, left = top * 4, right * 4, bottom * 4, left * 4

            # Compare against known faces
            distances = face_recognition.face_distance(known_encodings, face_encoding)
            best_match_idx = np.argmin(distances)
            best_distance = distances[best_match_idx]

            if best_distance < threshold:
                name = known_names[best_match_idx]
                confidence = (1 - best_distance) * 100
                color = (0, 255, 0)   # Green for known
                label = f'{name} ({confidence:.0f}%)'
            else:
                name = 'Unknown'
                color = (0, 0, 255)   # Red for unknown
                label = name

            # Draw bounding box
            cv2.rectangle(frame, (left, top), (right, bottom), color, 2)

            # Draw label background + text
            cv2.rectangle(frame, (left, bottom - 30), (right, bottom), color, cv2.FILLED)
            cv2.putText(frame, label, (left + 6, bottom - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

        # Show face count
        cv2.putText(frame, f'Faces detected: {len(face_locations)}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

        cv2.imshow('Real-Time Face Recognition', frame)

        key = cv2.waitKey(1) & 0xFF
        if key == 27:   # ESC key
            break
        elif key == ord('s'):   # Save snapshot
            snapshot_count += 1
            snap_path = f'snapshot_{snapshot_count}.jpg'
            cv2.imwrite(snap_path, frame)
            print(f'[INFO] Saved {snap_path}')

    video.release()
    cv2.destroyAllWindows()
    print('[SUCCESS] Webcam session ended.')


# Run it!
run_realtime_recognition(known_encodings, known_names, threshold=0.6)